In [1]:
import h5py
import numpy as np
from pathlib import Path

DATASET_DIR = Path("./dataset_v3")
SAMPLING_ID = 2
GROUP_SIZE = 32
TIME_HORIZON = 10000.0
ICS_PATH = DATASET_DIR / f"ICs_sampling_{SAMPLING_ID}.hdf5"
TOKENIZED_ICS_PATH = DATASET_DIR / f"ICs_sampling_{SAMPLING_ID}_groupof{GROUP_SIZE}.hdf5"
LABELS_PATH = DATASET_DIR / f"completed_systems_sampling_{SAMPLING_ID}.txt"

print("Inspecting:", ICS_PATH)
with h5py.File(ICS_PATH, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP] {name}")
    f.visititems(show)


Inspecting: dataset_v3/ICs_sampling_2.hdf5
[DATASET] APs shape=(160000, 2) dtype=float64
[DATASET] LANs shape=(160000, 2) dtype=float64
[DATASET] aratios shape=(160000, 1) dtype=float64
[DATASET] e_uvecs shape=(160000, 2, 3) dtype=float64
[DATASET] eccs shape=(160000, 2) dtype=float64
[DATASET] h_uvecs shape=(160000, 2, 3) dtype=float64
[DATASET] incs shape=(160000, 2) dtype=float64
[DATASET] masses shape=(160000, 3) dtype=float64
[DATASET] mratios shape=(160000, 2) dtype=float64
[DATASET] periods shape=(160000, 2) dtype=float64
[DATASET] positions shape=(160000, 3, 3) dtype=float64
[DATASET] q_uvecs shape=(160000, 2, 3) dtype=float64
[DATASET] rel_positions shape=(160000, 2, 3) dtype=float64
[DATASET] rel_velocities shape=(160000, 2, 3) dtype=float64
[DATASET] smaxes shape=(160000, 2) dtype=float64
[DATASET] tAnos shape=(160000, 2) dtype=float64
[DATASET] velocities shape=(160000, 3, 3) dtype=float64


In [2]:
# Regroup the sampling_2 initial conditions into 32-way variation sets.
input_path = ICS_PATH
output_path = TOKENIZED_ICS_PATH

token_size = GROUP_SIZE
tokenized = {}
features = ["aratios", "mratios", "eccs", "incs", "APs", "LANs"]

with h5py.File(input_path, "r") as fin:
    for name, obj in fin.items():
        if not isinstance(obj, h5py.Dataset):
            continue
        if name not in features:
            continue
        x = obj[:]
        n = (x.shape[0] // token_size) * token_size
        x_cut = x[:n]
        x_tok = x_cut.reshape(-1, token_size, *x.shape[1:])
        tokenized[name] = x_tok
        print(f"{name}: {x.shape} -> {x_tok.shape}")

with h5py.File(output_path, "w") as fout:
    for name, arr in tokenized.items():
        fout.create_dataset(name, data=arr)

print("Token regroup done ->", output_path)


APs: (160000, 2) -> (5000, 32, 2)
LANs: (160000, 2) -> (5000, 32, 2)
aratios: (160000, 1) -> (5000, 32, 1)
eccs: (160000, 2) -> (5000, 32, 2)
incs: (160000, 2) -> (5000, 32, 2)
mratios: (160000, 2) -> (5000, 32, 2)
Token regroup done -> dataset_v3/ICs_sampling_2_groupof32.hdf5


In [3]:
import numpy as np

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians -> (..., 2k) = [sin(x), cos(x)]"""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

tokenized["APs"]  = sincos(tokenized["APs"])    # (5000, 8, 2) -> (5000, 8, 4)
tokenized["LANs"] = sincos(tokenized["LANs"])   # (5000, 8, 2) -> (5000, 8, 4)
tokenized["incs"] = sincos(tokenized["incs"])   # (5000, 8, 2) -> (5000, 8, 4)

print("APs :", tokenized["APs"].shape)
print("LANs:", tokenized["LANs"].shape)
print("incs:", tokenized["incs"].shape)
print("write the data into the save file ready for the folloing")
with h5py.File(output_path, "w") as fout:
    for name, arr in tokenized.items():
        fout.create_dataset(name, data=arr)
        print(f"{name}: {arr.shape}")

APs : (5000, 32, 4)
LANs: (5000, 32, 4)
incs: (5000, 32, 4)
write the data into the save file ready for the folloing
APs: (5000, 32, 4)
LANs: (5000, 32, 4)
aratios: (5000, 32, 1)
eccs: (5000, 32, 2)
incs: (5000, 32, 4)
mratios: (5000, 32, 2)


In [4]:
import pandas as pd
import numpy as np
import torch

df = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None,
                 names=["index", "num_batch", "time_unbound", "wall_time", "diagnosis"])

df = df.astype({"index": "int64", "num_batch": "int64", "time_unbound": "float32", "wall_time": "float32", "diagnosis": "int64"})
df = df.sort_values("index").reset_index(drop=True)
df["raw_case_id"] = df["index"] // GROUP_SIZE
df["variation_id"] = df["index"] % GROUP_SIZE

valid_mask = (df["time_unbound"] >= 0) & (df["diagnosis"] == 0)
df_valid = df.loc[valid_mask].copy()
case_counts = df_valid["raw_case_id"].value_counts().sort_index()
complete_case_ids = case_counts[case_counts == GROUP_SIZE].index.to_numpy()
df_valid = df_valid[df_valid["raw_case_id"].isin(complete_case_ids)].copy()
df_valid["label"] = (df_valid["time_unbound"] >= TIME_HORIZON).astype(np.int32)

group_stability = (
    df_valid.groupby("raw_case_id")["label"]
      .max()
      .reset_index(name="is_unstable")
)
print(df_valid.columns)
print(df_valid.head(10))
print(group_stability.head())
print("valid complete cases:", len(group_stability))


Index(['index', 'num_batch', 'time_unbound', 'wall_time', 'diagnosis',
       'raw_case_id', 'variation_id', 'label'],
      dtype='object')
   index  num_batch  time_unbound   wall_time  diagnosis  raw_case_id  \
0      0          0       10000.0  128.617844          0            0   
1      1          0       10000.0  126.132133          0            0   
2      2          0       10000.0  127.196609          0            0   
3      3          0       10000.0  127.244606          0            0   
4      4          0       10000.0  129.052979          0            0   
5      5          0       10000.0  128.769531          0            0   
6      6          0       10000.0  127.129807          0            0   
7      7          0       10000.0  126.864037          0            0   
8      8          0       10000.0  127.569229          0            0   
9      9          0       10000.0  125.175858          0            0   

   variation_id  label  
0             0      1  
1    

In [5]:
import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# 1) Y: unstable label per group
#    1 if any of the 32 trajectories survives to the horizon threshold.
def build_group_unstable_labels(
    labels_path=LABELS_PATH,
    token_size=GROUP_SIZE,
    threshold_time=TIME_HORIZON,
):
    df = pd.read_csv(
        labels_path, sep=r"\s+", header=None,
        names=["index", "num_batch", "time_unbound", "wall_time", "diagnosis"]
    )
    df = df.astype({
        "index": "int64", "num_batch": "int64", "time_unbound": "float32",
        "wall_time": "float32", "diagnosis": "int64"
    })
    df = df.sort_values("index").reset_index(drop=True)
    df["raw_case_id"] = df["index"] // token_size

    valid_mask = (df["time_unbound"] >= 0) & (df["diagnosis"] == 0)
    df = df.loc[valid_mask].copy()
    case_counts = df["raw_case_id"].value_counts().sort_index()
    complete_case_ids = case_counts[case_counts == token_size].index.to_numpy(dtype=np.int64)
    df = df[df["raw_case_id"].isin(complete_case_ids)].copy()
    df["label"] = (df["time_unbound"] >= threshold_time).astype(np.float32)

    y_group = df.groupby("raw_case_id")["label"].max().to_numpy(dtype=np.float32)
    return y_group, complete_case_ids


# 2) X: take only 1 IC per group
#    from (N, 32, ...) -> (N, ...)
def load_tokenized_X_take_one(
    h5_path=TOKENIZED_ICS_PATH,
    keys=None,
    take_token_idx=0,
    group_ids=None,
    dtype=np.float32,
):
    with h5py.File(h5_path, "r") as f:
        all_keys = [k for k in f.keys() if isinstance(f[k], h5py.Dataset)]
        if keys is None:
            keys = all_keys
        else:
            missing = [k for k in keys if k not in f]
            if missing:
                raise KeyError(f"Missing keys in hdf5: {missing}. Available: {all_keys}")

        X_parts = []
        N = None
        for k in keys:
            arr = f[k][:]
            if arr.ndim < 2:
                raise ValueError(f"{k} has shape {arr.shape}, expected at least 2D (N,32,...)")
            if N is None:
                N = arr.shape[0]
            elif arr.shape[0] != N:
                raise ValueError(f"{k} has N={arr.shape[0]} but expected N={N}")
            if arr.shape[1] <= take_token_idx:
                raise ValueError(f"{k} second dim is {arr.shape[1]}, cannot take idx={take_token_idx}")

            one = arr[:, take_token_idx, ...].astype(dtype, copy=False)
            if group_ids is not None:
                one = one[np.asarray(group_ids, dtype=np.int64)]
            one_flat = one.reshape(one.shape[0], -1)
            X_parts.append(one_flat)

        X = np.concatenate(X_parts, axis=1)
    return X, keys


def stratified_holdout_indices(y, holdout_size, rng):
    y = np.asarray(y).astype(np.int64)
    idx = np.arange(len(y))
    pos_idx = idx[y == 1]
    neg_idx = idx[y == 0]

    n_pos = int(round(holdout_size * len(pos_idx) / len(y)))
    n_neg = holdout_size - n_pos
    n_pos = min(n_pos, len(pos_idx))
    n_neg = min(n_neg, len(neg_idx))

    remain = holdout_size - (n_pos + n_neg)
    if remain > 0:
        if len(pos_idx) - n_pos >= remain:
            n_pos += remain
        else:
            n_neg += remain

    holdout_pos = rng.choice(pos_idx, size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=np.int64)
    holdout_neg = rng.choice(neg_idx, size=n_neg, replace=False) if n_neg > 0 else np.array([], dtype=np.int64)

    holdout_idx = np.concatenate([holdout_pos, holdout_neg])
    rng.shuffle(holdout_idx)

    keep_mask = np.ones(len(y), dtype=bool)
    keep_mask[holdout_idx] = False
    remain_idx = idx[keep_mask]
    return remain_idx, holdout_idx


def split_train_test(X, y, test_size=1000, seed=42):
    assert X.shape[0] == y.shape[0]
    rng = np.random.default_rng(seed)
    train_idx, test_idx = stratified_holdout_indices(y, test_size, rng)
    return train_idx, test_idx


def standardize_train_test(X_train, X_test, eps=1e-6):
    mu = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.maximum(std, eps)
    return (X_train - mu) / std, (X_test - mu) / std, mu, std


def make_loaders(X_train, y_train, X_test, y_test, batch_size=128):
    X_train_t = torch.from_numpy(X_train).float()
    y_train_t = torch.from_numpy(y_train).float().unsqueeze(1)
    X_test_t  = torch.from_numpy(X_test).float()
    y_test_t  = torch.from_numpy(y_test).float().unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


def binary_metrics_from_probs(y_true, y_prob):
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= 0.5).astype(np.int32)
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan")
    f1 = f1_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else float("nan")
    acc = accuracy_score(y_true, y_pred)
    return {"auc": auc, "f1": f1, "acc": acc}


# MAIN
y, valid_group_ids = build_group_unstable_labels(LABELS_PATH, token_size=GROUP_SIZE, threshold_time=TIME_HORIZON)
X, used_keys = load_tokenized_X_take_one(TOKENIZED_ICS_PATH, keys=None, take_token_idx=0, group_ids=valid_group_ids)

assert X.shape[0] == y.shape[0]

print("X:", X.shape, "y:", y.shape, "keys_used:", len(used_keys))

train_idx, test_idx = split_train_test(X, y, test_size=1000, seed=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

X_train, X_test, mu, std = standardize_train_test(X_train, X_test)

train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test, batch_size=128)

print("Train:", X_train.shape, y_train.shape, " Test:", X_test.shape, y_test.shape)
print("y_train unstable stats:", float(y_train.min()), float(y_train.mean()), float(y_train.max()))
print("y_test  unstable stats:", float(y_test.min()), float(y_test.mean()), float(y_test.max()))


X: (4998, 17) y: (4998,) keys_used: 6
Train: (3998, 17) (3998,)  Test: (1000, 17) (1000,)
y_train unstable stats: 0.0 0.6615808010101318 1.0
y_test  unstable stats: 0.0 0.6610000133514404 1.0


## mlp

In [6]:
import torch
import torch.nn as nn


class MLPBinaryClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden=(512, 256, 128), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def eval_mlp_classifier(model, loader, device):
    model.eval()
    all_prob, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            prob = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            all_prob.append(prob)
            all_y.append(yb.numpy().ravel())

    all_prob = np.concatenate(all_prob)
    all_y = np.concatenate(all_y)
    return binary_metrics_from_probs(all_y, all_prob)


def train_mlp_classifier(
    train_loader,
    in_dim: int,
    device=None,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=30,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MLPBinaryClassifier(in_dim=in_dim).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            train_loss += loss.item() * bs
            n_train += bs

        train_loss /= max(n_train, 1)
        print(f"epoch {ep:03d} | train_bce {train_loss:.6f}")

    return model


def print_loader_metrics(model, loader, device, prefix):
    metrics = eval_mlp_classifier(model, loader, device)
    print(f"{prefix} | AUC {metrics['auc']:.4f} | F1 {metrics['f1']:.4f} | Acc {metrics['acc']:.4f}")
    return metrics


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = X_train.shape[1]
model = train_mlp_classifier(train_loader, in_dim=in_dim, device=device, epochs=500)

train_metrics = print_loader_metrics(model, train_loader, device, "TRAIN")
test_metrics = print_loader_metrics(model, test_loader, device, "TEST")

epoch 001 | train_bce 0.565233
epoch 002 | train_bce 0.506534
epoch 003 | train_bce 0.486878
epoch 004 | train_bce 0.476507
epoch 005 | train_bce 0.460297
epoch 006 | train_bce 0.452622
epoch 007 | train_bce 0.450942
epoch 008 | train_bce 0.439710
epoch 009 | train_bce 0.422631
epoch 010 | train_bce 0.412937
epoch 011 | train_bce 0.402499
epoch 012 | train_bce 0.394975
epoch 013 | train_bce 0.374880
epoch 014 | train_bce 0.385853
epoch 015 | train_bce 0.359865
epoch 016 | train_bce 0.350040
epoch 017 | train_bce 0.348276
epoch 018 | train_bce 0.347781
epoch 019 | train_bce 0.316629
epoch 020 | train_bce 0.321635
epoch 021 | train_bce 0.301593
epoch 022 | train_bce 0.295468
epoch 023 | train_bce 0.287511
epoch 024 | train_bce 0.279475
epoch 025 | train_bce 0.278457
epoch 026 | train_bce 0.269683
epoch 027 | train_bce 0.259138
epoch 028 | train_bce 0.251493
epoch 029 | train_bce 0.234227
epoch 030 | train_bce 0.241455
epoch 031 | train_bce 0.238792
epoch 032 | train_bce 0.219711
epoch 03

## XGBoost

In [8]:
# xgb_binary_classifier.py
# XGBoost binary classifier for predicting instability

from __future__ import annotations
import numpy as np

try:
    import xgboost as xgb
except ImportError as e:
    raise ImportError("Missing dependency: xgboost. Install with: pip install xgboost") from e

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, f1_score


# ---------------------------
# Utilities
# ---------------------------
def loader_to_numpy(loader):
    """
    Convert a PyTorch DataLoader yielding (xb, yb) into numpy arrays.

    Expected:
        xb: (B, D)
        yb: (B, 1) or (B,)

    Returns:
        X: (N, D) float32
        y: (N,) float32
    """
    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb.detach().cpu().numpy())
        ys.append(yb.detach().cpu().numpy())

    X = np.concatenate(xs, axis=0).astype(np.float32)
    y = np.concatenate(ys, axis=0).reshape(-1).astype(np.float32)
    return X, y

def check_xy(X: np.ndarray, y: np.ndarray, name="data"):
    X = np.asarray(X)
    y = np.asarray(y)

    if X.ndim != 2:
        raise ValueError(f"{name}: X must be 2D (N, D), got shape {X.shape}")
    if y.ndim not in (1, 2):
        raise ValueError(f"{name}: y must be 1D or 2D, got shape {y.shape}")

    y = y.reshape(-1)
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"{name}: X has {X.shape[0]} rows but y has {y.shape[0]} rows")

    if not np.isfinite(X).all():
        raise ValueError(f"{name}: X contains NaN or inf")
    if not np.isfinite(y).all():
        raise ValueError(f"{name}: y contains NaN or inf")

    return X.astype(np.float32), y.astype(np.float32)


# ---------------------------
# Model
# ---------------------------
class XGBBinaryClassifier:
    """
    Predict whether a group is unstable.

    Input:
        X: (N, D)

    Output:
        pred_proba: (N, 1), probability of the positive class
    """
    def __init__(
        self,
        n_estimators: int = 500,
        learning_rate: float = 0.05,
        max_depth: int = 6,
        subsample: float = 0.8,
        colsample_bytree: float = 0.8,
        reg_lambda: float = 1.0,
        reg_alpha: float = 0.0,
        min_child_weight: float = 1.0,
        gamma: float = 0.0,
        random_state: int = 42,
        tree_method: str = "hist",
        n_jobs: int = -1,
    ):
        self.model = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric=["logloss", "auc"],
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
            min_child_weight=min_child_weight,
            gamma=gamma,
            random_state=random_state,
            tree_method=tree_method,
            n_jobs=n_jobs,
        )

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray | None = None,
        y_val: np.ndarray | None = None,
        verbose: bool = True,
    ):
        X_train, y_train = check_xy(X_train, y_train, name="train")

        eval_set = None
        if X_val is not None and y_val is not None:
            X_val, y_val = check_xy(X_val, y_val, name="val")
            eval_set = [(X_train, y_train), (X_val, y_val)]

        self.model.fit(
            X_train,
            y_train,
            eval_set=eval_set,
            verbose=verbose,
        )
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 2:
            raise ValueError(f"X must be 2D (N, D), got shape {X.shape}")

        pred = self.model.predict_proba(X)[:, 1]
        return pred.reshape(-1, 1)


# ---------------------------
# Evaluation
# ---------------------------
def evaluate_binary_classifier(y_true: np.ndarray, y_pred_proba: np.ndarray, prefix=""):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.asarray(y_pred_proba).reshape(-1)

    y_pred = (y_pred_proba >= 0.5).astype(np.float32)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred) if np.unique(y_true).size >= 2 else float("nan")
    loss = log_loss(y_true, np.clip(y_pred_proba, 1e-7, 1 - 1e-7))
    if np.unique(y_true).size >= 2:
        auc = roc_auc_score(y_true, y_pred_proba)
        print(f"{prefix}auc {auc:.6f} | f1 {f1:.6f} | acc {acc:.6f} | logloss {loss:.6f}")
        return {"auc": auc, "f1": f1, "acc": acc, "logloss": loss}

    print(f"{prefix}auc skipped (single class) | f1 {f1:.6f} | acc {acc:.6f} | logloss {loss:.6f}")
    return {"auc": None, "f1": f1, "acc": acc, "logloss": loss}



# ---------------------------
# Training from arrays
# ---------------------------
def train_xgb_classifier_from_arrays(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    X_val: np.ndarray | None = None,
    y_val: np.ndarray | None = None,
    **xgb_kwargs,
):
    X_train, y_train = check_xy(X_train, y_train, name="train")
    X_test, y_test = check_xy(X_test, y_test, name="test")

    if X_val is not None and y_val is not None:
        X_val, y_val = check_xy(X_val, y_val, name="val")

    model = XGBBinaryClassifier(**xgb_kwargs)
    model.fit(
        X_train,
        y_train,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    print("XGB | TRAIN:", end=" ")
    evaluate_binary_classifier(y_train, pred_train)

    print("XGB | TEST :", end=" ")
    evaluate_binary_classifier(y_test, pred_test)

    return model


# ---------------------------
# Training from PyTorch loaders
# ---------------------------
def train_xgb_classifier_from_loaders(
    train_loader,
    test_loader,
    val_loader=None,
    **xgb_kwargs,
):
    X_train, y_train = loader_to_numpy(train_loader)
    X_test, y_test = loader_to_numpy(test_loader)

    if val_loader is not None:
        X_val, y_val = loader_to_numpy(val_loader)
    else:
        X_val, y_val = None, None

    return train_xgb_classifier_from_arrays(
        X_train, y_train,
        X_test, y_test,
        X_val=X_val, y_val=y_val,
        **xgb_kwargs,
    )

In [9]:
model = train_xgb_classifier_from_loaders(
    train_loader,
    test_loader,
    val_loader=None,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

XGB | TRAIN: auc 1.000000 | f1 1.000000 | acc 1.000000 | logloss 0.074268
XGB | TEST : auc 0.846224 | f1 0.849138 | acc 0.790000 | logloss 0.476539


In [10]:
import xgboost
print(xgboost.__version__)

1.7.6
